¿Qué porcentaje de partidos gana el local en cada temporada? ¿Y el visitante?
¿Cuántos goles marca el local y el visitante por partido, temporada a temporada?
La pregunta central: ¿cambió eso en la 2020/21 respecto a las temporadas con público? ¿Cuánto?
¿Se recuperó la ventaja cuando volvió el público en 2021/22?
¿Hay algo más que cambie sin público? Mira las tarjetas y las faltas: ¿sigue el árbitro favoreciendo al local cuando no hay 50.000 personas gritándole?

In [41]:
import pandas as pd
import csv

df1 = pd.read_csv('data/E0_1819.csv',usecols=['Date', 'HomeTeam', 'AwayTeam','FTHG', 'FTAG', 'FTR','HF','AF','HY','AY','HR','AR'])
df2 = pd.read_csv('data/E0_1920.csv',usecols=['Date', 'HomeTeam', 'AwayTeam','FTHG', 'FTAG', 'FTR','HF','AF','HY','AY','HR','AR'])
df3 = pd.read_csv('data/E0_2021.csv',usecols=['Date', 'HomeTeam', 'AwayTeam','FTHG', 'FTAG', 'FTR','HF','AF','HY','AY','HR','AR'])
df4 = pd.read_csv('data/E0_2122.csv',usecols=['Date', 'HomeTeam', 'AwayTeam','FTHG', 'FTAG', 'FTR','HF','AF','HY','AY','HR','AR'])
df5 = pd.read_csv('data/E0_2223.csv',usecols=['Date', 'HomeTeam', 'AwayTeam','FTHG', 'FTAG', 'FTR','HF','AF','HY','AY','HR','AR'])
df6 = pd.read_csv('data/E0_2324.csv',usecols=['Date', 'HomeTeam', 'AwayTeam','FTHG', 'FTAG', 'FTR','HF','AF','HY','AY','HR','AR'])

df_unido = pd.concat([df1, df2, df3, df4, df5, df6], keys=['18/19', '19/20', '20/21', '21/22', '22/23', '23/24'], names=['Temporada'])
df_final = df_unido.reset_index(level=0).reset_index(drop=True)
df_final['Date'] = pd.to_datetime(df_final['Date'], format='%d/%m/%Y')

condiciones = [
    (df_final['Temporada'] == '19/20') & (df_final['Date'] <= '2020-03-13'),
    (df_final['Temporada'] == '19/20') & (df_final['Date'] > '2020-03-13'),
]

resultados = ['19/20 (Antes del parón)', '19/20 (Después del parón)']


df_final['Temporadas'] = np.select(
    condiciones, resultados, default=df_final['Temporada']
)

¿Qué porcentaje de partidos gana el local en cada temporada? ¿Y el visitante?

In [42]:
partidos_ganados = df_final.groupby('Temporadas')['FTR'].value_counts(normalize=True)
print(partidos_ganados)

Temporadas                 FTR
18/19                      H      0.476316
                           A      0.336842
                           D      0.186842
19/20 (Antes del parón)    H      0.447917
                           A      0.302083
                           D      0.250000
19/20 (Después del parón)  H      0.467391
                           A      0.315217
                           D      0.217391
20/21                      A      0.402632
                           H      0.378947
                           D      0.218421
21/22                      H      0.428947
                           A      0.339474
                           D      0.231579
22/23                      H      0.484211
                           A      0.286842
                           D      0.228947
23/24                      H      0.460526
                           A      0.323684
                           D      0.215789
Name: proportion, dtype: float64


¿Cuántos goles marca el local y el visitante por partido, temporada a temporada?

In [43]:
goles_partido = df_final.groupby('Temporadas')[['FTHG','FTAG']].mean().round(3)
goles_partido['ventaja'] = (
    goles_partido['FTHG'] - goles_partido['FTAG']
)
print(goles_partido)

                            FTHG   FTAG  ventaja
Temporadas                                      
18/19                      1.568  1.253    0.315
19/20 (Antes del parón)    1.507  1.215    0.292
19/20 (Después del parón)  1.543  1.174    0.369
20/21                      1.353  1.342    0.011
21/22                      1.513  1.305    0.208
22/23                      1.634  1.218    0.416
23/24                      1.800  1.479    0.321


¿cambió el comportamiento arbitral sin público?

In [58]:
tarjetas_partido = df_final.groupby('Temporadas')[['HF','AF','HY','AY']].mean().round(3)
print(tarjetas_partido)

                               HF      AF     HY     AY
Temporadas                                             
18/19                      10.153  10.305  1.526  1.684
19/20 (Antes del parón)    10.278  10.903  1.628  1.875
19/20 (Después del parón)  11.130  11.228  1.457  1.424
20/21                      11.224  10.550  1.424  1.447
21/22                      10.055  10.158  1.653  1.745
22/23                      10.597  10.932  1.671  1.916
23/24                      10.745  11.347  1.911  2.263


In [60]:
goles_partido = df_final.groupby('Temporadas')[['FTHG','FTAG']].mean().round(3)
goles_partido['ventaja'] = (
    goles_partido['FTHG'] - goles_partido['FTAG']
)
print(goles_partido)

                            FTHG   FTAG  ventaja
Temporadas                                      
18/19                      1.568  1.253    0.315
19/20 (Antes del parón)    1.507  1.215    0.292
19/20 (Después del parón)  1.543  1.174    0.369
20/21                      1.353  1.342    0.011
21/22                      1.513  1.305    0.208
22/23                      1.634  1.218    0.416
23/24                      1.800  1.479    0.321


Con publico vs sin publico

In [100]:

df_final['ventaja_partido'] = df_final['FTHG'] - df_final['FTAG']

condicion_sin_publico = (df_final['Date'] >= '2020-03-13') & (
    df_final['Date'] <= '2021-05-31'
)

df_final['Condicion_Publico'] = 'Con Público'
df_final.loc[condicion_sin_publico, 'Condicion_Publico'] = 'Sin Público'

tabla = (
    df_final.groupby('Condicion_Publico')['ventaja_partido']
    .agg(n='count', Ventaja='mean', σ='std')
    .reset_index()
)

tabla['Error estándar'] = tabla['σ'] / (tabla['n'] ** 0.5)
tabla['IC 95%'] = tabla['Error estándar'] * 2

tabla_formateada = pd.DataFrame()
tabla_formateada['Grupo'] = tabla['Condicion_Publico']
tabla_formateada['n'] = tabla['n'].apply(lambda x: f'{x:,}'.replace(',', '.'))
tabla_formateada['Ventaja'] = tabla['Ventaja'].apply(lambda x: f'+{x:.3f}')
tabla_formateada['σ'] = tabla['σ'].apply(lambda x: f'{x:.3f}')
tabla_formateada['Error estándar'] = tabla['Error estándar'].apply(
    lambda x: f'±{x:.3f}'
)
tabla_formateada['IC 95%'] = tabla['IC 95%'].apply(lambda x: f'±{x:.3f}')

tabla_formateada = tabla_formateada.sort_values(
    by='Grupo', ascending=False
).reset_index(drop=True)

print(tabla_formateada.to_string(index=False))

      Grupo     n Ventaja     σ Error estándar IC 95%
Sin Público   472  +0.081 1.940         ±0.089 ±0.179
Con Público 1.808  +0.311 1.956         ±0.046 ±0.092


Esta selección mide los partidos sin público comparados con los partidos con público. Los partidos sin público son tomados desde el parón en 2020-03-13, hasta el 2021-05-31. Son en total 1808 partidos con público y 472 sin público. El error estandar es de ±0.046 en el primer grupo y ±0.089 en el segundo. Queda una zona gris, en diciembre de 2020 algunos estadios ingleses admitieron un par de miles de aficionados durante unas jornadas, y en la 21/22 hubo restricciones puntuales.

¿Qué porcentaje de partidos gana el local en cada temporada? ¿Y el visitante?

In [95]:
partidos_ganados = df_final.groupby('Condicion_Publico')['FTR'].value_counts(normalize=True)
print(partidos_ganados)

Condicion_Publico  FTR
Con Público        H      0.460177
                   A      0.318584
                   D      0.221239
Sin Público        H      0.396186
                   A      0.385593
                   D      0.218220
Name: proportion, dtype: float64


¿cambió el comportamiento arbitral sin público?

In [96]:
tarjetas_partido = df_final.groupby('Condicion_Publico')[['HF','AF','HY','AY']].mean().round(3)
print(tarjetas_partido)

                       HF      AF    HY     AY
Condicion_Publico                             
Con Público        10.370  10.720  1.68  1.898
Sin Público        11.206  10.682  1.43  1.443


¿Cuántos goles marca el local y el visitante por partido, temporada a temporada?

In [97]:
goles_partido = df_final.groupby('Condicion_Publico')[['FTHG','FTAG']].mean().round(3)
goles_partido['ventaja'] = (
    goles_partido['FTHG'] - goles_partido['FTAG']
)
print(goles_partido)

                   FTHG   FTAG  ventaja
Condicion_Publico                      
Con Público        1.61  1.298    0.312
Sin Público        1.39  1.309    0.081


Resultado final

In [103]:

df_final['Ventaja_Partido'] = df_final['FTHG'] - df_final['FTAG']

resumen = (
    df_final.groupby('Condicion_Publico')['Ventaja_Partido']
    .agg(['count', 'mean', 'std'])
    .reset_index()
)

# Calculamos el error estándar de cada grupo (desviación / raíz del conteo) y lo añadimos a la tabla
resumen['error_estandar'] = resumen['std'] / (resumen['count'] ** 0.5)

print('--- RESUMEN POR GRUPO CON ERROR ESTÁNDAR ---')
print(
    resumen[
        ['Condicion_Publico', 'count', 'mean', 'std', 'error_estandar']
    ].to_string(index=False)
)

# Extraemos los valores para calcular el margen de la diferencia entre los dos grupos
n1 = resumen.loc[resumen['Condicion_Publico'] == 'Con Público', 'count'].values[0]
m1 = resumen.loc[resumen['Condicion_Publico'] == 'Con Público', 'mean'].values[0]
s1 = resumen.loc[resumen['Condicion_Publico'] == 'Con Público', 'std'].values[0]
se1 = resumen.loc[
    resumen['Condicion_Publico'] == 'Con Público', 'error_estandar'
].values[0]

n2 = resumen.loc[
    resumen['Condicion_Publico'] == 'Sin Público', 'count'
].values[0]
m2 = resumen.loc[resumen['Condicion_Publico'] == 'Sin Público', 'mean'].values[0]
s2 = resumen.loc[resumen['Condicion_Publico'] == 'Sin Público', 'std'].values[0]
se2 = resumen.loc[
    resumen['Condicion_Publico'] == 'Sin Público', 'error_estandar'
].values[0]

#Cálculo del margen de la diferencia entre los dos grupos
dif_medias = m1 - m2
se_dif = ((s1**2 / n1) + (s2**2 / n2)) ** 0.5
margen_dif = se_dif * 2  # Margen de incertidumbre al 95% para la diferencia

ic_inf = dif_medias - margen_dif
ic_sup = dif_medias + margen_dif

print('\n--- MARGEN DE LA DIFERENCIA ENTRE GRUPOS ---')
print(f'Error estándar (Con Público): ±{se1:.3f}')
print(f'Error estándar (Sin Público): ±{se2:.3f}')
print(f'Diferencia de medias: {dif_medias:.3f}')
print(f'Margen de la diferencia (IC 95%): ±{margen_dif:.3f}')
print(
    f'Intervalo de confianza: entre {ic_inf:.3f} y {ic_sup:.3f}. Como no incluye el cero, no es azar.'
)

--- RESUMEN POR GRUPO CON ERROR ESTÁNDAR ---
Condicion_Publico  count     mean      std  error_estandar
      Con Público   1808 0.311394 1.955748        0.045995
      Sin Público    472 0.080508 1.940156        0.089303

--- MARGEN DE LA DIFERENCIA ENTRE GRUPOS ---
Error estándar (Con Público): ±0.046
Error estándar (Sin Público): ±0.089
Diferencia de medias: 0.231
Margen de la diferencia (IC 95%): ±0.201
Intervalo de confianza: entre 0.030 y 0.432. Como no incluye el cero, no es azar.
